<a href="https://colab.research.google.com/github/LikithaSrinivasa25/ml-dl-algorithms/blob/main/nlp/NLP_Seq2Seq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

In [ ]:
from keras.models import Model
from keras.layers import Input, LSTM, Dense, SimpleRNN

In [ ]:
batch_size = 64
epochs =10
latent_dim = 256
num_samples  = 10000

In [ ]:
data = "/content/tel.txt"

input_texts = []
target_texts =[]
input_characters = set()
target_characters = set()

with open(data, "r", encoding = "utf-8") as f:
  lines=f.read().split("\n")
for line in lines[: min(num_samples, len(lines)-1)]:
  input_text, target_text, unwanted = line.split("\t")
  target_text = "\t" + target_text + "\n"
  input_texts.append(input_text)
  target_texts.append(target_text)
  for char in input_text:
    if char not in input_characters:
      input_characters.add(char) # Set of all English Alphabets A-Z/a-z and punctuations
  for char in target_text:
    if char not in target_characters:
      target_characters.add(char) # Set of all telugu Alphabets



In [ ]:
input_texts[:5]

['Wonderful!', 'Definitely!', 'He hung up.', 'I ran home.', 'Who are we?']

In [ ]:
target_texts[:5]

['\tఅద్భుతం!\n',
 '\tతప్పకుండా\n',
 '\tఅతను పెట్టేసాడు\n',
 '\tనేను ఇంటికి పరిగెత్తాను\n',
 '\tమేము ఎవరము ?\n']

In [ ]:
print('Number of samples:', len(input_texts))

Number of samples: 152


In [ ]:
input_characters =sorted(list(input_characters))
target_characters = sorted(list(target_characters))
num_encoder_tokens = len(input_characters)
num_decoder_tokens = len(target_characters)
print("Number of Unique Input Tokens: ", num_encoder_tokens)
print("Number of Unique Output Tokens: ", num_decoder_tokens)

Number of Unique Input Tokens:  50
Number of Unique Output Tokens:  56


In [ ]:
max_encoder_seq_length =max([len(txt) for txt in input_texts])
max_decoder_seq_length =max([len(txt) for txt in target_texts])
print("Maximum Sequence Input Length: ", max_encoder_seq_length)
print("Maximum Sequence Output  Length: ", max_decoder_seq_length)


Maximum Sequence Input Length:  60
Maximum Sequence Output  Length:  63


In [ ]:
input_token_index =dict([(char, i) for i, char in enumerate(input_characters)])
target_token_index =dict([(char,i) for i, char in enumerate(target_characters)])

In [ ]:
encoder_input_data = np.zeros((len(input_texts), max_encoder_seq_length, num_encoder_tokens), dtype = "float32")
decoder_input_data = np.zeros((len(input_texts), max_decoder_seq_length, num_decoder_tokens), dtype = "float32")
decoder_target_data = np.zeros((len(input_texts), max_decoder_seq_length, num_decoder_tokens), dtype= "float32")

In [ ]:
for i, (input_text, target_text) in enumerate(zip(input_texts, target_texts)):
  for t, char in enumerate(input_text):
    encoder_input_data[i, t, input_token_index[char]]=1.
  encoder_input_data[i, t+1:, input_token_index[" "]]=1.
  for t, char in enumerate(target_text):
    decoder_input_data[i, t, target_token_index[char]]=1.
    if t>0:
      decoder_target_data[i, t-1, target_token_index[char]]
  decoder_input_data[i, t+1: , target_token_index[char]]
  decoder_target_data[i, t:, target_token_index[" "]]=1.


In [ ]:
encoder_inputs =Input(shape=(None, num_encoder_tokens))
encoder = LSTM(latent_dim, return_state =True)
encoder_outputs, state_h , state_c = encoder(encoder_inputs)
encoder_states = [state_h, state_c]
decoder_inputs = Input(shape=(None, num_decoder_tokens))
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state =True)
decoder_outputs, _, _ =decoder_lstm(decoder_inputs, initial_state=encoder_states)
decoder_dense = Dense(num_decoder_tokens, activation ="softmax")
decoder_outputs =decoder_dense(decoder_outputs)
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

In [ ]:
model.compile(optimizer = "rmsprop", loss ="categorical_crossentropy")
model.fit([encoder_input_data, decoder_input_data], decoder_target_data, batch_size=batch_size, epochs=epochs, validation_split = 0.2)

Epoch 1/10
2/2 [==============================] - 7s 2s/step - loss: 2.2873 - val_loss: 0.5353
Epoch 2/10
2/2 [==============================] - 2s 730ms/step - loss: 0.4914 - val_loss: 0.1481
Epoch 3/10
2/2 [==============================] - 1s 620ms/step - loss: 0.1731 - val_loss: 0.1279
Epoch 4/10
2/2 [==============================] - 1s 665ms/step - loss: 0.2026 - val_loss: 0.0338
Epoch 5/10
2/2 [==============================] - 1s 613ms/step - loss: 0.0974 - val_loss: 0.0261
Epoch 6/10
2/2 [==============================] - 1s 641ms/step - loss: 0.1014 - val_loss: 0.0806
Epoch 7/10
2/2 [==============================] - 1s 641ms/step - loss: 0.1669 - val_loss: 0.0608
Epoch 8/10
2/2 [==============================] - 1s 634ms/step - loss: 0.1219 - val_loss: 0.0309
Epoch 9/10
2/2 [==============================] - 1s 647ms/step - loss: 0.1032 - val_loss: 0.0652
Epoch 10/10
2/2 [==============================] - 1s 647ms/step - loss: 0.1392 - val_loss: 0.0231


In [ ]:
encoder_model = Model(encoder_inputs, encoder_states)

decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]
decoder_outputs, state_h, state_c = decoder_lstm(decoder_inputs, initial_state=decoder_states_inputs)
decoder_states = [state_h, state_c]
decoder_outputs = decoder_dense(decoder_outputs)
decoder_model = Model([decoder_inputs] + decoder_states_inputs,[decoder_outputs] + decoder_states)

In [ ]:
reverse_input_char_index = dict((i,char) for char, i in input_token_index.items())
reverse_target_char_index =dict((i,char) for char, i in target_token_index.items())

In [ ]:
def decode_sequence(input_seq):
  states_value= encoder_model.predict(input_seq, verbose=0)
  target_seq =np.zeros((1,1, num_decoder_tokens))
  target_seq[0,0, target_token_index["\t"]]=1.0
  stop_condition=False
  decoded_sentence=""

  while not stop_condition:
    output_tokens, h,c =decoder_model.predict([target_seq]+states_value, verbose = 0)
    sampled_token_index = np.argmax(output_tokens[0,-1,:])
    sampled_char =reverse_target_char_index[sampled_token_index]
    decoded_sentence+=sampled_char
    if (sampled_char =="\n" or len(decoded_sentence)>max_decoder_seq_length):
      stop_condition =True

    target_seq = np.zeros((1, 1, num_decoder_tokens))
    target_seq[0, 0, sampled_token_index] = 1.0
    states_value = [h, c]
  return decoded_sentence

In [ ]:

for seq_index in range(5):
    input_seq = encoder_input_data[seq_index: seq_index + 1]
    decoded_sentence = decode_sequence(input_seq)
    print('-')
    print('Input sentence:', input_texts[seq_index])
    print('Decoded sentence:', decoded_sentence)

-
Input sentence: Wonderful!
Decoded sentence:                                                                 
-
Input sentence: Definitely!
Decoded sentence:                                                                 
-
Input sentence: He hung up.
Decoded sentence:                                                                 
-
Input sentence: I ran home.
Decoded sentence:                                                                 
-
Input sentence: Who are we?
Decoded sentence:                                                                 
